# Load ERA5-Land Data

In [1]:
# auto reload modified modules
%load_ext autoreload
%autoreload 2
import sys
WORK_DIR = "/home/research/jenzheng/documents/kai/research/eta"
sys.path.append(WORK_DIR)
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from plot_utils import plot_setting, plot_nfields
from utils import *
import torch
import torch.nn as nn
from kde import get_data_pdf
plot_setting()

# Load the cropped data
file_path = WORK_DIR + '/data/ERA5/era5land_USA_SouthEast_1999-2023_dailymax.nc'
ds = xr.open_dataset(file_path, engine='netcdf4')

# Extract the 'tp' variable (Total Precipitation)
tp = ds['tp']

# Convert to a NumPy array and multiply by 1000 (meter to millimeter)
tp_numpy = tp.values * 1000
# 
# Check if there are any NaN values
has_nan = np.isnan(tp_numpy).any()
# print(f"Contains NaN values: {has_nan}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Downsample Data

In [2]:
# Downsampling by a factor of ds_fact for each spatial dimension
ds_fact = 10
tp_ds_numpy = tp_numpy[:, ::ds_fact, ::ds_fact]  # (9050, 40, 80)

# Data Preprocessing

In [ ]:
max_values_original = np.max(tp_numpy, axis=(1, 2))

# max_values_original will have shape (9050,)
print(max_values_original.shape)

# Sort max_values and get the indices that would sort the array
sorted_indices_original = np.argsort(max_values_original)
print(0 in sorted_indices_original)

# Use the sorted_indices to get the sorted max_values
sorted_max_values_original = max_values_original[sorted_indices_original]

# trim the tail
trim_tail_thresh = 240
num_trim_days = len(sorted_max_values_original[sorted_max_values_original > trim_tail_thresh])
trim_days = sorted_indices_original[-num_trim_days:]
sorted_max_values_trim = sorted_max_values_original[:-num_trim_days]
sorted_indices_trim = sorted_indices_original[:-num_trim_days]
print("Trimmed Days: ", trim_days)
print("Trimmed Values: ", sorted_max_values_original[-num_trim_days:])

kept_days = np.delete(np.arange(len(tp_numpy)), trim_days)
tp_trim_numpy = tp_numpy[kept_days]
tp_trim_ds_numpy = tp_ds_numpy[kept_days]
tp_trim_tensor = torch.tensor(tp_trim_numpy)
tp_trim_ds_tensor = torch.tensor(tp_trim_ds_numpy)
print(tp_trim_ds_tensor.shape)
print(tp_trim_tensor.shape)

# Select Training Data

In [ ]:
import torch.optim as optim
from rich.progress import Progress
from torch.utils.data import DataLoader, TensorDataset
from copy import deepcopy
import os
import gc

# Convert numpy arrays to PyTorch tensors and move to GPU
tp_trim_ds_tensor = torch.tensor(tp_trim_ds_numpy, dtype=torch.float32).unsqueeze(1).to(device)  # Add channel dimension
tp_trim_tensor = torch.tensor(tp_trim_numpy, dtype=torch.float32).unsqueeze(1).to(device)  # Full-resolution target

# Use 20% of the data (5 years) for training
num_years = 0.5
portion = num_years/25
train_size = int(portion * len(tp_trim_ds_tensor))
train_input = tp_trim_ds_tensor[:train_size]
train_target = tp_trim_tensor[:train_size]

# use full data to select the best MSE model for baseline comparison
test_input = deepcopy(tp_trim_ds_tensor)
test_target = deepcopy(tp_trim_tensor)

# move data to device
train_input = train_input.to(device)
train_target = train_target.to(device)
test_input = test_input.to(device)
test_target = test_target.to(device)

# Create DataLoader for batch processing
batch_size = 64
train_dataset = TensorDataset(train_input, train_target)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataset = TensorDataset(test_input, test_target)
test_loader = DataLoader(test_dataset, batch_size=25*batch_size, shuffle=False)

print(train_input.shape)
print(train_target.shape)
print(test_input.shape)
print(test_target.shape)

# Start Training MSE

In [ ]:
from models import SRCNN
seed = 43
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Initialize CNN model, optimizer, and loss function
model_mse = SRCNN(hidden_dim=64, num_blocks=3, scale_factor=ds_fact)

# Move model and data to multiple GPUs
model_mse = nn.DataParallel(model_mse)  # Use all available GPUs
model_mse.to(device)

# optimizer and loss
optimizer = optim.Adam(model_mse.parameters(), lr=3e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=200, gamma=.2)
mse_loss = nn.MSELoss()

# Training settings
num_epochs = 500
grad_clip = 1

# Checkpoint
save_mse = False
best_loss = float('inf')
save_dir = WORK_DIR + "/models/precip-srcnn/"
os.makedirs(save_dir, exist_ok=True)
model_mse_filename = f"srcnn-mse-{num_years}yr-{ds_fact}ds.pth"
save_path_mse = os.path.join(save_dir, model_mse_filename)
print(save_path_mse)

with Progress() as progress:
    print("Training Started")
    epoch_task = progress.add_task("Training Epochs", total=num_epochs)

    for epoch in range(num_epochs):
        running_train_loss = 0.0
        running_test_loss = 0.0
        # Progress bar for each epoch
        # batch_task = progress.add_task(f"Epoch {epoch+1}/{num_epochs}", total=len(train_loader))
        
        model_mse.train()
        for batch_idx, (input_data, target) in enumerate(train_loader):
            model_mse.train()
            
            # Zero the gradients
            optimizer.zero_grad()
            
            # Forward pass
            output = model_mse(input_data)

            # Compute loss
            loss = mse_loss(output, target)
            
            # Backward pass
            loss.backward()
        
            # Update weights
            optimizer.step()

            # Update loss
            running_train_loss += loss.item()
        
        epoch_train_loss = running_train_loss/len(train_loader)
        scheduler.step()
        
        model_mse.eval()
        with torch.no_grad():
            for _, (test_in, test_tar) in enumerate(test_loader):
                test_output = model_mse(test_in)
                loss = mse_loss(test_output, test_tar)
                running_test_loss += loss.item()
                
            epoch_test_loss = running_test_loss/len(test_loader)
            if epoch_test_loss < best_loss:
                best_loss = epoch_test_loss
                if save_mse:
                    torch.save(model_mse.state_dict(), save_path_mse)
            
        # Update epoch progress bar
        progress.update(epoch_task, advance=1)
        progress.console.print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {epoch_train_loss:.6f}, Best Val Loss: {best_loss:.6f}")#, Average Upsample Loss : {upsample_loss/len(train_loader):.6f}")


# Load Trained MSE Model (If Applicable)

In [ ]:
# load path
load_dir = WORK_DIR + "/models/precip-srcnn/"
os.makedirs(load_dir, exist_ok=True)

# model name
model_mse_filename = f"srcnn-mse-{num_years}yr-{ds_fact}ds.pth"
print(model_mse_filename)
# save model
load_path = os.path.join(load_dir, model_mse_filename)

# load model
model_mse = SRCNN(hidden_dim=64, num_blocks=3, scale_factor=ds_fact)
model_mse = nn.DataParallel(model_mse)
model_mse.load_state_dict(torch.load(load_path))
model_mse.to(device)
print()

# Run MSE-Model on Trimmed Dataset

In [ ]:
from test_utils import test_model_mse
test_dataset = TensorDataset(test_input, test_target)
test_loader = DataLoader(test_dataset, batch_size=50*batch_size, shuffle=False)

full_output_mse, test_mse_loss_mse = test_model_mse(model_mse, test_loader)
print(full_output_mse.shape)
print("MSE Test Loss:", test_mse_loss_mse)

# MSE Output PDF

In [ ]:
max_values = np.max(tp_trim_numpy, axis=(1, 2))
max_values_mse = np.max(full_output_mse, axis=(1,2))
y_eval_tp, py_tp, _ = get_data_pdf(data=torch.tensor(max_values), data_all=torch.tensor(max_values))
y_eval_tp_mse, py_tp_mse, _ = get_data_pdf(data=torch.tensor(max_values_mse), data_all=torch.tensor(max_values_mse))

# Plotting the two KDEs for comparison
plt.figure(figsize=(10, 6), dpi=400)

# Plot the KDE for the original data
plt.plot(y_eval_tp, py_tp, label=r"$y_\#\mu$")
# plt.plot(y_eval_tp_train, py_tp_train, label="5 Years True")
plt.plot(y_eval_tp_mse, py_tp_mse, label=r"$y_{\xi\#}\mu$")

# Add plot labels and title
plt.xlabel("Total Precipitation (mm)")
plt.ylabel("Log Density")
plt.title("Log Scale", fontsize=18)
plt.ylim(1e-5,1e-1)
plt.yscale('log')
plt.legend(prop={'size':18})

# Fit EVD

Parameters of GEV: **shape** $k$, **location** $\mu$, and **scale** $\sigma > 0$
$$f(x; k, \mu, \sigma) = \frac{1}{\sigma} \left(1 + k \frac{x - \mu}{\sigma}\right)^{-(1 + \frac{1}{k})} \exp\left(-\left(1 + k \frac{x - \mu}{\sigma}\right)^{-1/k}\right)$$

1. **Gumbel Distribution** ($k = 0$):
   $$
   f(x; \mu, \sigma) = \frac{1}{\sigma} \exp\left(-\frac{x - \mu}{\sigma} - \exp\left(-\frac{x - \mu}{\sigma}\right)\right)
   $$

2. **Frechet Distribution** ($k > 0$): Heavy-tailed.

3. **Weibull Distribution** ($k < 0$): Bounded upper tail.

In [13]:
from scipy.stats import genextreme
from scipy.integrate import quad

def fit_gev_with_cutoff(data, cutoff):
    """
    Fits a Generalized Extreme Value (GEV) distribution to the given 1D data,
    with a cutoff applied. All values greater than the cutoff are ignored,
    and the density is normalized.

    Parameters:
        data (array-like): The 1D data array to which the GEV distribution is fitted.
        cutoff (float): The cutoff value. Data values above this are ignored.

    Returns:
        dict: A dictionary containing the shape, location, scale parameters,
              and additional information about the fitting process.
    """
    # Filter the data based on the cutoff
    filtered_data = data[data <= cutoff]

    # Ensure filtered data is not empty
    if len(filtered_data) == 0:
        raise ValueError("No data points are below the cutoff.")

    # Fit the GEV distribution to the filtered data
    shape, loc, scale = genextreme.fit(filtered_data)
    
    scale = scale + 4 # manually lift up the tail a bit
    normalization_constant = quad(
                            lambda t: genextreme.pdf(t, shape, loc=loc, scale=scale),
                            -np.inf, cutoff)[0]

    # Define the normalized PDF with cutoff
    def normalized_pdf(x):
        raw_pdf = genextreme.pdf(x, shape, loc=loc, scale=scale)
        return raw_pdf / normalization_constant
    
    x = np.linspace(min(filtered_data), cutoff, len(data))
    pdf = np.array([normalized_pdf(val) for val in x])

    # Results as a dictionary
    results = {
        "shape": shape,
        "location": loc,
        "scale": scale,
        "cutoff": cutoff,
        "c": normalization_constant,
        "x": x,
        "pdf": pdf
    }

    return results

cutoff_value = np.ceil(max(max_values)) + 20
fit_results = fit_gev_with_cutoff(max_values, cutoff=cutoff_value)

In [ ]:
plt.figure(figsize=(10, 6), dpi=400)
# plt.hist(max_values, bins=40, density=True, alpha=0.4, color="gray", label="25 Years True")
# plt.plot(x, pdf, "r-", lw=2, label="Normalized GEV PDF")
plt.plot(y_eval_tp, py_tp, label="25 Years True")
plt.plot(fit_results['x'], fit_results['pdf'], "r-", lw=2, label="GEVD")
plt.xlabel("Data")
plt.ylabel("Density")
plt.yscale('log')
plt.legend()

plt.show()

# Quantile Estimation
$$Q(p ; \mu, \sigma, k)= \begin{cases}\mu-\sigma \ln (-\ln p) & \text { for } k=0 \text { and } p \in(0,1) \\ \mu+\frac{\sigma}{k}\left((-\ln p)^{-k}-1\right) & \text { for } k>0 \text { and } p \in[0,1), \text { or } k<0 \text { and } p \in(0,1] ;\end{cases}$$

In [ ]:
tau = 0.95 # quantile
y_tau = genextreme.ppf(tau*fit_results['c'], fit_results['shape'], fit_results['location'], fit_results['scale'])
Qy = lambda q : genextreme.ppf(q*fit_results['c'], fit_results['shape'], fit_results['location'], fit_results['scale'])
gamma = 1- fit_results['c']

print(y_tau)
print(gamma)
print(fit_results['cutoff'])
print(fit_results['shape'])
print(fit_results['scale'])

# ETA with W1 Loss

In [ ]:
# tail threshold, equivalent to \tau in the paper
w1_tail_thresh = y_tau

# sort max values
max_values = np.max(tp_trim_numpy, axis=(1,2))
sorted_indices = np.argsort(max_values)
sorted_max_values = max_values[sorted_indices]

# get tail that defines w1
# num_w1_days = len(sorted_max_values[sorted_max_values > w1_tail_thresh])
num_w1_days = 350
w1_truedays = sorted_indices[-num_w1_days:]

quantiles = torch.linspace(tau, 1, num_w1_days)
w1_truemax = torch.tensor([Qy(q) for q in quantiles])
print(w1_truemax)

def get_fixed_days_max_pos(full_output: np.ndarray, w1_days):
    """
    helper function - get the location index of the tail values 
    
    input:
        - selected days: 
        
    """
    max_indices = np.argmax(full_output.reshape(full_output.shape[0], -1), axis=1)
    max_pos_row, max_pos_col = np.unravel_index(max_indices, (80,160))
    max_pos = np.stack([np.arange(full_output.shape[0]), max_pos_row, max_pos_col], axis=1).T
    
    num_w1_days = len(w1_days)
    fixed_days_max_pos = torch.tensor(max_pos[:,w1_days])
    fixed_days_max_pos = torch.cat((torch.arange(num_w1_days).reshape(1,-1), fixed_days_max_pos[1:,:]), dim=0)
    fixed_days_max_pos.requires_grad = False
    
    return fixed_days_max_pos

def get_max_time_pos(full_output : np.ndarray, num_w1_days):
    """
    helper function - get the time and location index of the tail values 
    
    input:
        - full_output: output of model running on the entire testing dataset
        - num_w1_days: number of tail days taken, equivalent to n_q (number of 
                       pre-determined quantile levels) in the paper
    """
    # get w1_days
    max_values_output = np.max(full_output, axis=(1,2))
    sorted_indices_output = np.argsort(max_values_output)
    w1_days = sorted_indices_output[-num_w1_days:]
    
    # get w1_max_pos
    max_indices = np.argmax(full_output.reshape(full_output.shape[0], -1), axis=1)
    max_pos_row, max_pos_col = np.unravel_index(max_indices, (80,160))
    max_pos = np.stack([np.arange(full_output.shape[0]), max_pos_row, max_pos_col], axis=1).T
    w1_max_pos = torch.tensor(max_pos[:,w1_days])
    w1_max_pos = torch.cat((torch.arange(num_w1_days).reshape(1,-1), w1_max_pos[1:,:]), dim=0)
    w1_max_pos.requires_grad = False
    
    return w1_days, w1_max_pos
    

# def get_w1_objects_dynamic_days():
#     pass

# w1_truedays_max_pos = get_fixed_days_max_pos(full_output_mse, w1_truedays)
# w1_days, w1_max_pos = get_max_time_pos(full_output_mse, num_w1_days)
print(num_w1_days)

In [ ]:
def get_w1_input(w1_days):
    """
    function to get w1 loss input
    """
    w1_input = tp_trim_ds_numpy[w1_days]
    w1_input = torch.tensor(w1_input).unsqueeze(1)
    return w1_input
    
num_mse_years = num_years
mse_portion = num_mse_years/25
mse_size = int(mse_portion * len(tp_trim_ds_tensor))
mse_input = tp_trim_ds_tensor[:mse_size]
mse_target = tp_trim_tensor[:mse_size]

print(mse_input.shape)
print(mse_target.shape)

# move data to device
w1_truemax = w1_truemax.to(device)
mse_input = mse_input.to(device)
mse_target = mse_target.to(device)

# ETA Testing Function

In [19]:
def test_model_eta(model, TestLoader, 
                   w1_loss, w1_input, w1_truemax, w1_max_pos, 
                   device=None):
    """
    run model on TestLoader and report the MSE loss and the W1 loss
    
    return:
        - full_output: squeezed numpy array with dimension (N,...)
        - test_loss_mse: mean squared error on TestLoader
    
    """
    if not device:
        device = next(model.parameters()).device
    model.to(device)
    model.eval()
    _, sample_target = next(iter(TestLoader))
    full_output = torch.zeros_like(sample_target[0]).cpu()
    mse_loss = nn.MSELoss()
    running_test_mse_loss = 0.0
    
    with torch.no_grad():
        for _, (test_in, test_tar) in enumerate(test_loader):
            test_in, test_tar = test_in.to(device), test_tar.to(device)  
            test_output = model(test_in).detach()
            loss = mse_loss(test_output, test_tar)
            running_test_mse_loss += loss.item()
            full_output = torch.cat((full_output,test_output.squeeze().cpu()),dim=0)
    test_mse_loss = running_test_mse_loss/len(test_loader)
    
    # when testing, should record the true tail values
    num_w1_days = len(w1_truemax)
    full_output = full_output[1:].numpy()
    max_values_output = np.max(full_output, axis=(1,2))
    sorted_indices_output = np.argsort(max_values_output)
    sorted_max_values_output = max_values_output[sorted_indices_output]
    w1_max_test = torch.tensor(sorted_max_values_output[-num_w1_days:])
    w1_max_test = w1_max_test.to(w1_truemax.device)
    
#     w1_output_test = model(w1_input).squeeze()
#     w1_max_test = w1_output_test[tuple(w1_max_pos)]
    w1_loss_test = w1_loss(w1_max_test, w1_truemax).item()
    
    return full_output, test_mse_loss, w1_loss_test

def get_max_indices_vals(full_output : np.ndarray, num_w1_days):
    max_values_output = np.max(full_output, axis=(1,2))
    sorted_indices_output = np.argsort(max_values_output)
    sorted_max_values_output = max_values_output[sorted_indices_output]
    max_indices = sorted_indices_output[-num_w1_days:]
    max_vals = torch.tensor(sorted_max_values_output[-num_w1_days:])
    return max_indices, max_vals

# ETA Training Function

In [20]:
def train_model_eta(model_eta, test_loader, 
                    seed, num_epochs,
                    mse_input, mse_target,
                    w1_loss, w1_truemax, w1_truedays,
                    lambd_, omega:int, varying_days:bool,
                    save_checkpoint=True, save_path=None,
                    scheduler=None, clip_gradient=False,
                    device=None):
    """
    Training function of \eta-learning
    
    Inputs:
        - model_eta: pretrained model with MSE
        - lambd_: loss balancing parameter
        - omega: window size 
    
    """
    
    global_seed(seed)
    if not device:
        device = next(model_eta.parameters()).device
    
    if save_checkpoint and not save_path:
        raise ValueError("No Path to Save Model!")
    
    # define mse_loss
    mse_loss = nn.MSELoss()
    
    # move data to device
    w1_truemax = w1_truemax.to(device)
    mse_input = mse_input.to(device)
    mse_target = mse_target.to(device)
    
    # for checkpoint use
    best_mse_loss = float('inf')
    best_w1_loss = float('inf')
    best_total_loss = float('inf')
    
    # record training loss history
    train_mse_loss_hist = []
    train_w1_loss_hist = []
    test_mse_loss_hist = []
    test_w1_loss_hist = []
    
    # get initialization of w1_days + w1_max_pos
    num_w1_days = len(w1_truedays)
    full_output_eta, _ = test_model_mse(model_eta, test_loader)
    w1_days = None
    if varying_days:
        w1_days, w1_max_pos = get_max_time_pos(full_output_eta, num_w1_days)
    else:
        w1_max_pos = get_fixed_days_max_pos(full_output_eta, w1_truedays)
        w1_days = w1_truedays
    w1_input = get_w1_input(w1_days)
    
    with Progress() as progress:
        epoch_task = progress.add_task("Training Epochs", total=num_epochs)

        for epoch in range(num_epochs):
            # could add minibatch training, but no need for this test case
            model_eta.train()

            # Zero the gradients
            optimizer.zero_grad()
            
            # get w1_input, move to device
            if varying_days:
                w1_input = get_w1_input(w1_days)
            w1_input = w1_input.to(device)
            
            # Forward pass
            mse_output = model_eta(mse_input)
            w1_output = model_eta(w1_input).squeeze()
            w1_max = w1_output[tuple(w1_max_pos)]

            # Compute loss
            mse_loss_val = mse_loss(mse_output, mse_target)
            w1_loss_val = w1_loss(w1_max, w1_truemax)
            
            # Backward pass
            loss = mse_loss_val + lambd_ * w1_loss_val
            loss.backward()

            # Gradient clipping
            if clip_gradient:
                torch.nn.utils.clip_grad_norm_(model_eta.parameters(), grad_clip)

            # Update weights
            optimizer.step()
            if scheduler:
                scheduler.step()
            
            # Run inference on more inputs
            full_output_eta, test_mse_loss_eta, w1_loss_test = test_model_eta(
                model_eta,
                test_loader,
                w1_loss,
                w1_input,
                w1_truemax,
                w1_max_pos,
                device=device
            )
            
            # update w1_max_pos whenever window size is reached
            if (epoch+1)%omega == 0:
                if varying_days:
                    w1_days, w1_max_pos = get_max_time_pos(full_output_eta, num_w1_days)
                else:
                    # here, w1_days = w1_truedays
                    w1_max_pos = get_fixed_days_max_pos(full_output_eta, w1_days)

            # save best w1 loss model
            if w1_loss_test < best_w1_loss:
                best_w1_loss = w1_loss_test
                best_mse_loss = test_mse_loss_eta
                if save_checkpoint:
                    torch.save(model_eta.state_dict(), save_path)

            # save loss history
            train_mse_loss_hist.append(mse_loss_val)
            train_w1_loss_hist.append(w1_loss_val)
            test_mse_loss_hist.append(test_mse_loss_eta)
            test_w1_loss_hist.append(w1_loss_test)
            
            # Update epoch progress bar
            progress.update(epoch_task, advance=1)
            progress.console.print(
                f"Epoch {epoch+1}/{num_epochs}, "
#                 f"Best MSE Test Loss: {best_mse_loss:.6f}, "
#                 f"Best W1 Test Loss: {best_w1_loss:.6f}, "
#                 f"Curr W1 Test Loss: {w1_loss_test:.6f}"
                f"Best MSE: {best_mse_loss:.6f}, "
                f"Best W1: {best_w1_loss:.6f}, "
                f"Curr W1: {w1_loss_test:.6f}, "
            )
#             progress.console.print(f"Epoch {epoch+1}/{num_epochs}, Best MSE Test Loss: {best_mse_loss:.6f}, Best W1 Test Loss: {best_w1_loss:.6f}")
    
    loss_dict = {
        "train_mse" : train_mse_loss_hist,
        "train_w1" : train_w1_loss_hist,
        "test_mse" : test_mse_loss_hist,
        "test_w1" : test_w1_loss_hist
    }
    
    return model_eta, loss_dict

In [ ]:
# training components
seed = 43
global_seed(seed)
model_eta = deepcopy(model_mse).to(device)
num_epochs = 150
optimizer = optim.Adam(model_eta.parameters(), lr=3e-4)
w1_loss = lambda x,y: torch.abs(x-y).mean()
lambd_ = 1.
omega = 1 # window size
varying_days = True
test_dataset = TensorDataset(test_input, test_target)
test_loader = DataLoader(test_dataset, batch_size=50*batch_size, shuffle=False)

# training techniques
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=.6)
# clip_gradient = False

# checkpoint
save_eta = True
best_mse_loss = float('inf')
best_w1_loss = float('inf')
best_total_loss = float('inf')
save_dir = WORK_DIR + "/models/precip-srcnn/"
os.makedirs(save_dir, exist_ok=True)
model_eta_filename = f"srcnn-eta-gevd-{num_years}yr-{ds_fact}ds-{int(w1_tail_thresh)}tail-{omega}omega.pth"
save_path_eta = os.path.join(save_dir, model_eta_filename)
print(save_path_eta)

In [22]:
# model_eta, loss_dict_eta = train_model_eta(
#     model_eta, 
#     test_loader, 
#     seed,
#     num_epochs,
#     mse_input, 
#     mse_target,
#     w1_loss, 
#     w1_truemax, 
#     w1_truedays,
#     lambd_, 
#     omega, 
#     varying_days,
#     save_checkpoint=save_eta, 
#     save_path=save_path_eta
# #     scheduler=scheduler, 
# #     clip_gradient=clip_gradient,
# )

# Run ETA on Full Dataset

In [ ]:
test_dataset = TensorDataset(test_input, test_target)
test_loader = DataLoader(test_dataset, batch_size=25*batch_size, shuffle=False)

omega=1
w1_tail_thresh = w1_tail_thresh
save_dir = WORK_DIR + "/models/precip-srcnn/"
load_path_eta = os.path.join(save_dir, f"srcnn-eta-gevd-{num_years}yr-{ds_fact}ds-{int(w1_tail_thresh)}tail-{omega}omega.pth")
print(load_path_eta)
# device = "cuda:0"
# model_eta = CorrectNet(model_mse, scale_factor=ds_fact, kernel_size=3, padding=1)
model_eta = SRCNN(hidden_dim=64, num_blocks=3, scale_factor=ds_fact)
model_eta = nn.DataParallel(model_eta)
model_eta.load_state_dict(torch.load(load_path_eta))
model_eta.to(device)
full_output_eta, test_mse_loss_eta, w1_loss_test = test_model_eta(
    model_eta,
    test_loader,
    w1_loss,
    None,
    w1_truemax,
    None,
    device=device
)

print(full_output_eta.shape)
print("ETA MSE Test Loss:", test_mse_loss_eta)
print("ETA W1 Test Loss:", w1_loss_test)

In [ ]:
def get_max_indices_values(full_output : np.ndarray, num_w1_days):
    max_values_output = np.max(full_output, axis=(1,2))
    sorted_indices_output = np.argsort(max_values_output)
    sorted_max_values_output = max_values_output[sorted_indices_output]
    max_value_indices = sorted_indices_output[-num_w1_days:]
    max_values = torch.tensor(sorted_max_values_output[-num_w1_days:])
    return max_value_indices, max_values

max_value_indices_eta, max_values_eta = get_max_indices_values(full_output_eta, num_w1_days)
print(max_value_indices_eta)
print(w1_truedays)
print(max_values_eta)

In [25]:
def plot_pdf_eta(y, py, linestyle=None, color=None,
                 label=None, labelsize=16, 
                 title=None, titlesize=30, 
                 fig=None, ax=None, figsize=(12,5)):
    """ 
    inputs:
        y - ndarray, provided by get_output_pdf
        py - pdf value of array y
        label - if provided, figure will draw legend for the pdf curve
        title - suptitle
        fig, ax - if provided, will continue drawing on fig
    """
    if fig is None:
        fig, ax = plt.subplots(1,2,figsize=figsize,dpi=400)
    
    if fig is not None:
        assert ax is not None 
    if ax is not None:
        assert fig is not None

    if linestyle is not None:
        ax[0].plot(y, py, color=color, label=label, linestyle=linestyle)#, color='royalblue')
        ax[1].plot(y, py, color=color, linestyle=linestyle)#, color='royalblue')
    else:
        ax[0].plot(y, py, color=color, label=label)#, color='royalblue')
        ax[1].plot(y, py, color=color)#, color='royalblue')
        
    ax[0].set_title('Linear Scale', fontsize=24)
    ax[0].set_xlabel("Daily max precipitation (mm)", fontsize=14)

    ax[1].set_yscale('log')
    ax[1].set_title('Log Scale', fontsize=24)
    ax[1].set_xlabel("Daily max precipitation (mm)", fontsize=14)
    if title is not None:
        fig.suptitle(title, fontsize=titlesize)
    if label is not None:
        ax[0].legend(prop={'size':labelsize})

    return fig, ax

In [ ]:
# full PDF comparison
max_values_eta = np.max(full_output_eta, axis=(1,2))
max_values_train = np.max(tp_trim_numpy[:train_size], axis=(1,2))
# max_values_eta = max_values_eta[max_values_eta < sorted_max_values[-1]]
# tail_values_eta = max_values_eta[max_values_eta > w1_tail_thresh]

y_eval_true, py_true, _ = get_data_pdf(data=torch.tensor(max_values), data_all=torch.tensor(max_values))
y_eval_mse, py_mse, _ = get_data_pdf(data=torch.tensor(max_values_mse), data_all=torch.tensor(max_values_mse))
y_eval_eta, py_eta, _ = get_data_pdf(data=torch.tensor(max_values_eta), data_all=torch.tensor(max_values_eta))
y_eval_train, py_train, _ = get_data_pdf(data=torch.tensor(max_values_train), data_all=torch.tensor(max_values_train))

# Plotting the two KDEs for comparison
plt.figure(figsize=(10, 6), dpi=400)

# Plot the PDFs
marker_every = 100
plt.plot(y_eval_true, py_true, label=r"HR Ground Truth (25-Yr)", color='blue', linestyle='-', linewidth=2) 
plt.plot(y_eval_train, py_train, label=r"HR Training Data of $\eta$-Map (0.5-Yr)", color='orange', linestyle=':', 
         marker='d', markevery=100, linewidth=2)
plt.plot(fit_results['x'], fit_results['pdf'], label="Hypothesized Extreme Value Distribution", color='purple', linestyle=':', 
         marker='s', markevery=800, linewidth=2)
plt.plot(y_eval_mse, py_mse, label=r"HR MSE-Mapped Samples (0.5-Yr Training)", color='green', linestyle=':', 
         marker='^', markevery=marker_every, linewidth=2, markersize=10)
plt.plot(y_eval_eta, py_eta, label=r"HR $\eta$-Mapped Samples (0.5-Yr Training)", color='red', linestyle=':', 
         marker='*', markersize=14, markevery=marker_every, linewidth=3)
# plt.hist(max_values, bins=40, density=True, alpha=0.3, color="gray", label=r"$\{y_i\}_{i=1}^{9044}$")

# plot the KDE truncated tails
# plt.plot(y_eval_true[y_eval_true > 150], py_true[y_eval_true > 150], label="25 Years True")
# plt.plot(y_eval_mse[y_eval_mse > 150], py_mse[y_eval_mse > 150], label="25 Years MSE", linestyle='-.')
# plt.plot(y_eval_eta[y_eval_eta > 150], py_eta[y_eval_eta > 150], label=fr"25 Years $\eta$", linestyle='--')

# Plot the tails
# plt.plot(y_tail_eval_true, py_tail_true, label="25 Years True Tail")
# plt.plot(y_tail_eval_mse, py_tail_mse, label="25 Years MSE Tail", linestyle='-.')
# plt.plot(y_tail_eval_eta, py_tail_eta, label=fr"25 Years $\eta$ Tail", linestyle='--')

# Add plot labels and title
plt.xlabel("Maximal Daily Peak Precipitation (mm)", fontsize=14)
plt.ylabel("Log Density", fontsize=14)
# plt.title("PDF of Regional Max Precipitation", fontsize=14)
plt.ylim(1e-5,1e-1)
plt.yscale('log')
plt.legend(prop={'size': 14})

In [ ]:
visualize_day = 1364
time_str = str(tp.time.values[visualize_day])[:10]
ds_field = tp_trim_ds_numpy[visualize_day]
true_field = tp_trim_numpy[visualize_day]
mse_field = full_output_mse[visualize_day]
eta_field = full_output_eta[visualize_day]
fig,axes = plot_nfields(ds_field, true_field, mse_field, eta_field, title_date=time_str, longitudes=tp.longitude.values, latitudes=tp.latitude.values, add_ylabel=False)